# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Danyal-Nadeem/flyrank-ml-starter/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I'm picking the **provisional lane: Refresh / Content Opportunity Scoring** (Lane 2), not freestyle.

This lane is the cleanest fit for the four things a good research question has to connect — the data I have, the decision it improves, the action someone takes, and the cost of getting it wrong. The starter dataset already ships exactly the columns this lane needs (`trend_direction`, `impressions_90d`, `sessions_90d`, `avg_position`, `word_count`, `content_age_days`), and the starter pipeline has already been run on this exact lane — its verified output (`outputs/model_report.md`) shows a learned ranking clearly beats a hand-written rule at this problem, so I'm not starting from nothing. Lane 1 (Ranking Signal Analysis) stops at "what correlates" with no action attached; Lane 3 (Clustering) needs a harder-to-defend archetype-to-action mapping; Lane 4 (CTR Scoring) is really a narrower slice of this same refresh-decision problem. I can confirm or change this lane until end of Week 4, so this is a provisional, evidence-backed starting point.

In [9]:
chosen_lane = "Refresh / Content Opportunity Scoring (Lane 2)"
why = [
    "starter dataset already has the exact columns this lane needs",
    "starter pipeline already run on this lane with verified results",
    "other lanes lack a clear decision/action mapping",
]
print("Lane:", chosen_lane)
for r in why:
    print("-", r)

Lane: Refresh / Content Opportunity Scoring (Lane 2)
- starter dataset already has the exact columns this lane needs
- starter pipeline already run on this lane with verified results
- other lanes lack a clear decision/action mapping


## 2. The question: decision, action, cost of a wrong call

**Search question:** Given only the safe, observable signals in FlyRank's content dataset, which pages should be reviewed first for a content refresh — and how much can a learned ranking improve on a transparent hand-written rule at picking those pages?

**Unit of analysis:** one row = one content item (`content_id`), observed over the dataset's 90-day window.

**The decision this improves:** which pages a *limited-capacity* content review team looks at first this week, out of thousands of candidates.

**The action someone takes:** a reviewer opens a flagged page, checks its reason codes (e.g. `stale_visible_page`, `declining_with_demand`, `low_ctr_visible_page`), and decides whether to refresh, expand, protect, prune, or monitor it.

**The cost of a wrong recommendation:**
- *False positive* (flagged, but wasn't really declining): wastes a reviewer's limited hours on a page that didn't need attention.
- *False negative* (a genuinely declining, high-traffic page never gets flagged): worse — it keeps losing visibility/clicks silently until someone notices by accident, so real traffic is lost while nobody is looking.
- Because false negatives are more expensive here, a ranked queue (review top-K first) fits better than a single yes/no cutoff — it lets the team trade capacity against risk explicitly, using Precision@K rather than plain accuracy.

**Why data/ML can help at all:** a hand-written rule can only combine a few signals with fixed weights. A learned model can weigh many observable signals against each other, and has already been shown, on this exact dataset, to rank true candidates far more accurately than the fixed rule.

In [10]:
framing = {
    "decision": "which pages a capacity-limited review team looks at first",
    "actor": "content reviewer",
    "actions": ["refresh", "expand", "protect", "prune", "monitor"],
    "cost_false_positive": "wasted reviewer time on a page that wasn't declining",
    "cost_false_negative": "a real declining page keeps losing traffic unnoticed",
    "more_expensive_error": "false_negative",
    "metric_that_matches_this": "Precision@K (K = reviewer capacity, e.g. 50)",
}
for k, v in framing.items():
    print(f"{k}: {v}")

decision: which pages a capacity-limited review team looks at first
actor: content reviewer
actions: ['refresh', 'expand', 'protect', 'prune', 'monitor']
cost_false_positive: wasted reviewer time on a page that wasn't declining
cost_false_negative: a real declining page keeps losing traffic unnoticed
more_expensive_error: false_negative
metric_that_matches_this: Precision@K (K = reviewer capacity, e.g. 50)


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*
**Why this lane looks worth the next 7 weeks:** the starter dataset has 30,000 pages, and more than half of them — 16,262 (about 54%) — are already labeled `trend_direction == "down"`, the beginner proxy target for this lane. That's a large, non-trivial base rate: there's a real decline signal to rank against, not a rare-event problem. On top of that, the already-executed starter pipeline shows a learned model can rank these candidates far better than a hand-written rule (Precision@50: 0.240 baseline vs. 0.740 random forest — roughly 37 of the top 50 flagged pages correct instead of 12). Combined, that's both enough labeled signal to work with and already-measured evidence that a model beats the naive approach — a strong basis to build on for the next 7 weeks.

In [11]:
!git clone https://github.com/Danyal-Nadeem/flyrank-ml-starter.git
%cd flyrank-ml-starter/work/notebooks

Cloning into 'flyrank-ml-starter'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 119 (delta 36), reused 101 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 1.84 MiB | 12.00 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/flyrank-ml-starter/work/notebooks/flyrank-ml-starter/work/notebooks


In [12]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Total rows (pages) in starter dataset:", len(df))
print("\ntrend_direction value counts (beginner proxy label, trend_direction == 'down'):")
print(df["trend_direction"].value_counts())

eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
print("\nRows eligible after the pipeline's own filters (impressions_90d > 0 and content_age_days >= 90):", len(eligible))

Total rows (pages) in starter dataset: 30000

trend_direction value counts (beginner proxy label, trend_direction == 'down'):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Rows eligible after the pipeline's own filters (impressions_90d > 0 and content_age_days >= 90): 30000


## 4. Careful words: what I can and can't claim

**I can say:**
- These results are observed and directional, on a 30k-row anonymized starter slice, using client-holdout validation.
- The learned ranking outperforms the hand-written baseline on Precision@50 for this specific proxy label (`trend_direction == "down"`).
- This is decision-support: it ranks candidates for a human reviewer, not a guarantee of what happens if a page is refreshed.

**I cannot say:**
- That refreshing a flagged page *causes* recovery — that needs an experiment or causal design, which this data doesn't give me.
- That this proves anything about Google's actual ranking algorithm.
- That this result generalizes to the full ~79M-row warehouse — it has to be re-earned there with proper validation.
- That `trend_direction == "down"` is the ideal target — it's a current-window proxy label, not a future outcome. A stronger capstone version would move to a future-window label (e.g. prior 90 days → next 30 days decline).

In [13]:
claims = {
    "can_say": [
        "observed and directional, on the 30k-row starter slice",
        "learned ranking beats baseline on Precision@50 for this proxy label",
        "decision-support, not a guarantee of outcome",
    ],
    "cannot_say": [
        "refresh causes recovery (no causal design here)",
        "anything about Google's actual ranking algorithm",
        "generalizes to the full warehouse without re-validation",
        "trend_direction == 'down' is the ideal target label",
    ],
}
for k, vals in claims.items():
    print(k + ":")
    for v in vals:
        print("  -", v)

can_say:
  - observed and directional, on the 30k-row starter slice
  - learned ranking beats baseline on Precision@50 for this proxy label
  - decision-support, not a guarantee of outcome
cannot_say:
  - refresh causes recovery (no causal design here)
  - anything about Google's actual ranking algorithm
  - generalizes to the full warehouse without re-validation
  - trend_direction == 'down' is the ideal target label


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.